In [2]:
# homework 4 - valuation

In [3]:
# imports 

from tqdm.auto import tqdm

In [4]:
# generating data

In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

print(len(documents))
documents[0]

72


{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [6]:
# we want structured output and define structured data using pydantic:

from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]


# instructions for the llm: 
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [7]:
# initialize openai: 

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
# if something goes temporarily wrong e.g. due to a temporary or network issue, then retry

from evaluation_utils import llm_structured_retry

# we will se it in the processing function: 

def generate_ground_truth(doc):
    import json
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [9]:
# Parallel processing with 5 or 6 connections at once

# it is possible because we are just waiting for a response from openai
# we want to be a little carefult to not hit the provider limits somehow


# threadpoolexecutor allows us to have different workers, 
# we will have thread 1, thead 2, thread 3, ... each of these threads 
# will be an executor that is pulling of this common task pool. 

# then we iterate over this things, submit to the output executor, 
# thread pool executor is evaluating them and we get bakc the responses 

from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

# using it is very simple:
# we create this threat pool we say that we want at most 6 workders and then 
# we use this function

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(
        pool,
        documents,
        generate_ground_truth
    )

  0%|          | 0/72 [00:00<?, ?it/s]

In [10]:
results[0]

([{'question': 'What is RAG, and how does it help an LLM answer questions better?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'Why do we treat the language model as a black box in this course instead of building one ourselves?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What kinds of problems do LLMs have that make retrieval useful, like cutoff knowledge or hallucinations?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What are we building in this module, and what kind of FAQ example will it use?',
   'document': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What will be covered in the first part versus the second part of this RAG module?',
   'document': '01-agentic-rag/lessons/01-intro.md'}],
 ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=110, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_token

In [11]:
# split data and usage into different lists: 

# extent (everything)
# append (just one)
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

360

In [12]:
# calc usage price ... 
# price helper: 
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.1134

In [13]:
# turn all this into a pandas document so we can look at it and 
# also download it 

import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [14]:
df_ground_truth

,question,document
0,"What is RAG, and how does it help an LLM answe...",01-agentic-rag/lessons/01-intro.md
1,Why do we treat the language model as a black ...,01-agentic-rag/lessons/01-intro.md
2,What kinds of problems do LLMs have that make ...,01-agentic-rag/lessons/01-intro.md
3,"What are we building in this module, and what ...",01-agentic-rag/lessons/01-intro.md
4,What will be covered in the first part versus ...,01-agentic-rag/lessons/01-intro.md
...,...,...
355,How should I handle retrieval if my source is ...,07-project-example/lessons/07-chunking.md
356,What’s the recommended way to break up one lon...,07-project-example/lessons/07-chunking.md
357,"For books or other very long material, should ...",07-project-example/lessons/07-chunking.md
358,How can slide decks or images be turned into s...,07-project-example/lessons/07-chunking.md


In [15]:
df_ground_truth.to_csv("data/ground-truth-course-lessons.csv", index=False)

In [16]:
"""
Q1. Generating questions
Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/03-rag.md
Each call returns the token usage, which most LLM APIs report on the response object (e.g. response.usage.input_tokens / prompt_tokens).

What's the average number of input tokens across these 3 calls?

140
1400
14000
140000
These numbers vary between runs, even with the same model, so pick the closest option. A different provider or model may land further apart, but the input tokens stay in the same order of magnitude - the prompt we send is the same.
"""

"\nQ1. Generating questions\nGenerating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:\n\n01-agentic-rag/lessons/01-intro.md\n01-agentic-rag/lessons/02-environment.md\n01-agentic-rag/lessons/03-rag.md\nEach call returns the token usage, which most LLM APIs report on the response object (e.g. response.usage.input_tokens / prompt_tokens).\n\nWhat's the average number of input tokens across these 3 calls?\n\n140\n1400\n14000\n140000\nThese numbers vary between runs, even with the same model, so pick the closest option. A different provider or model may land further apart, but the input tokens stay in the same order of magnitude - the prompt we send is the same.\n"

In [17]:
# generate_ground_truth(doc)

total_input_tokens = 0

first_three_pages = documents[:3]
for page in first_three_pages:
    print("processing: ", page["filename"])
    results, usage = generate_ground_truth(page)
    total_input_tokens += usage.input_tokens
    print("Usage: ", usage)


print("total input tokens: ", total_input_tokens)
print("avg total_input_tokens: ", total_input_tokens / 3)


# SOLUTION: 1350.0 -> CLOSEST ANSWER: 1400


processing:  01-agentic-rag/lessons/01-intro.md
Usage:  ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=125, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1145)
processing:  01-agentic-rag/lessons/02-environment.md
Usage:  ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=125, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1411)
processing:  01-agentic-rag/lessons/03-rag.md
Usage:  ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=1280), output_tokens=100, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1853)
total input tokens:  4059
avg total_input_tokens:  1353.0


In [18]:
"""
The full ground truth
You don't need to generate the data for the rest of the homework. We already did it for all 72 pages, using the same approach as in the lessons, and saved the 360 questions to a file.

Download it:

PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
wget ${PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv
Load it with pandas into a dataframe of records called ground_truth. Each record has a question and the filename of the page that should answer it.
"""


# !PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
# !wget ${PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv

ground_truth = pd.read_csv("ground-truth.csv")
ground_truth.head(5)


,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md


In [19]:
ground_truth = ground_truth.to_dict(orient="records")

In [20]:
"""
Searching the chunks
We search over the same chunks as in homework 2.

Create them with chunk_documents:

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
This gives 295 chunks.

Now rebuild the search from homework 2 over these chunks. Build a text index (Index) and a vector index (VectorSearch), both keyed on filename. Wrap each one in a function, text_search and vector_search, that takes a query and the number of results to return (5 by default).

For hybrid search, reuse the rrf function from homework 2:

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]
Then define hybrid_search on top of it:

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)
"""


'\nSearching the chunks\nWe search over the same chunks as in homework 2.\n\nCreate them with chunk_documents:\n\nfrom gitsource import chunk_documents\n\nchunks = chunk_documents(documents, size=2000, step=1000)\nThis gives 295 chunks.\n\nNow rebuild the search from homework 2 over these chunks. Build a text index (Index) and a vector index (VectorSearch), both keyed on filename. Wrap each one in a function, text_search and vector_search, that takes a query and the number of results to return (5 by default).\n\nFor hybrid search, reuse the rrf function from homework 2:\n\ndef rrf(result_lists, k=60, num_results=5):\n    scores = {}\n    docs = {}\n\n    for results in result_lists:\n        for rank, doc in enumerate(results):\n            key = (doc["filename"], doc["start"])\n            scores[key] = scores.get(key, 0) + 1 / (k + rank)\n            docs[key] = doc\n\n    ranked = sorted(scores, key=scores.get, reverse=True)\n    return [docs[key] for key in ranked[:num_results]]\nT

In [21]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

# len = 295

In [22]:
chunks[:5]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [23]:
# Build a text index (Index), keyed on filename. Wrap in a function, that takes a query and the number of results to return (5 by default).


def build_index(documents):
    from minsearch import Index
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

index = build_index(documents=chunks)

def text_search(query, num_results=5):
    # the question boost is 3.0 if the user question matches exactly the faq then its more important
    # boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=num_results,
        # boost_dict=boost_dict
    )

example_text_search = text_search("What wheather is today?")
example_text_search

[{'start': 4000,
  'content': '\nCall the LLM for one document:\n\n```python\nfrom dotenv import load_dotenv\nfrom openai import OpenAI\n\nload_dotenv()\nopenai_client = OpenAI()\n```\n\nPrepare the document as JSON:\n\n```python\nimport json\n\nuser_prompt = json.dumps(doc)\n```\n\nCreate the messages:\n\n```python\nmessages = [\n    {"role": "developer", "content": data_gen_instructions},\n    {"role": "user", "content": user_prompt}\n]\n```\n\nUntil now we called `responses.create` and read `response.output_text`.\nFor structured output we switch to `responses.parse` and pass\n`text_format=Questions`, which tells the API to return our class instead\nof free text.\n\nCall the model:\n\n```python\nresponse = openai_client.responses.parse(\n    model="gpt-5.4-mini",\n    input=messages,\n    text_format=Questions\n)\n```\n\nThe parsed object is available in `response.output_parsed`:\n\n```python\nresult = response.output_parsed\n\nprint(result)\n```\n\nWe can access the list directly:\

In [24]:

# Build a vector index (VectorSearch), keyed on filename. Wrap in a function, that takes a query and the number of results to return (5 by default).

from embedder import Embedder
embed = Embedder() 

# embed lesson contents from chunks
contents = [
    chunk['content'] for chunk in chunks
]
vectors = embed.encode_batch(
    contents
)
import numpy as np
X = np.array(vectors)

from minsearch import VectorSearch
vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)


def vector_search(query, num_results=5):

    # embed query 
    v = embed.encode(query)

    return vindex.search(v)[:num_results]  # could be optimized with num_results keyword

example_vector_search = vector_search("What wheather is today?")
example_vector_search


[{'start': 4000,
  'content': 'relevance_total_text\n```\n\nFor the data we prepared on May 29, 2026, this gives:\n\n```python\n[\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach entry in `relevance_total_text` is a relevance list. This is\nenough to check that the function works before we run it for the full\ndataset.\n\nNext, make the relevance functions generic. We start with text search,\nbut later we may want to evaluate vector search, hybrid search, or\nanother retrieval method. The relevance logic is the same. Only the\nsearch function changes.\n\n```python\ndef compute_relevance(q, search_function):\n    doc_id = q["document"]\n    results = search_function(query=q["question"])\n\n    r

In [25]:
# For hybrid search, reuse the rrf function from homework 2:

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

# Then define hybrid_search on top of it:

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

example_hybrid_search = hybrid_search("What weather is today?")
example_hybrid_search

[{'start': 4000,
  'content': '\nCall the LLM for one document:\n\n```python\nfrom dotenv import load_dotenv\nfrom openai import OpenAI\n\nload_dotenv()\nopenai_client = OpenAI()\n```\n\nPrepare the document as JSON:\n\n```python\nimport json\n\nuser_prompt = json.dumps(doc)\n```\n\nCreate the messages:\n\n```python\nmessages = [\n    {"role": "developer", "content": data_gen_instructions},\n    {"role": "user", "content": user_prompt}\n]\n```\n\nUntil now we called `responses.create` and read `response.output_text`.\nFor structured output we switch to `responses.parse` and pass\n`text_format=Questions`, which tells the API to return our class instead\nof free text.\n\nCall the model:\n\n```python\nresponse = openai_client.responses.parse(\n    model="gpt-5.4-mini",\n    input=messages,\n    text_format=Questions\n)\n```\n\nThe parsed object is available in `response.output_parsed`:\n\n```python\nresult = response.output_parsed\n\nprint(result)\n```\n\nWe can access the list directly:\

In [26]:
"""
Q2. First result with text search
Take the first question from the ground truth:

q = ground_truth[0]["question"]
After running text_search for it, what's the filename of the first result?

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/10-rag-next-steps.md
"""

q = ground_truth[0]["question"]
print(q)
text_search(q)

# SOLUTION: filename': '01-agentic-rag/lessons/03-rag.md



What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [27]:
"""
Q3. First result with vector search
After running vector_search for the same question, what's the filename of the first result?

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/03-rag.md
04-evaluation/lessons/11-evaluation-intro.md
04-evaluation/lessons/12-rag-answers.md

This question was generated from 01-agentic-rag/lessons/01-intro.md. Notice that one method finds the right page at the top and the other doesn't. That's exactly why we measure across the whole dataset instead of trusting one query.
"""

q = ground_truth[0]["question"]
print(q)
vector_search(q)

# SOLUTION: 'filename': '01-agentic-rag/lessons/01-intro.md'


What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [28]:
"""
Evaluation metrics

We evaluate search exactly as in the module, reusing the same functions from the lecture. We change only the label. Our ground truth uses filename, so a result counts as a hit when a returned chunk's filename matches the question's filename, not a document id.

The lesson evaluates whole FAQ records, while this homework evaluates chunks. Adjust the relevance logic to use the fields available on the chunk results and homework ground truth instead of copying the lesson's record-based comparison unchanged.

As a reminder, these functions do the following:

- compute_relevance runs search for a question and returns a list of 0s and 1s
- hit_rate is the fraction of questions where the correct page appears in the results
- mrr (Mean Reciprocal Rank) also rewards finding the page near the top
- evaluate runs a search function over the whole ground truth and returns both metrics
"""

"\nEvaluation metrics\n\nWe evaluate search exactly as in the module, reusing the same functions from the lecture. We change only the label. Our ground truth uses filename, so a result counts as a hit when a returned chunk's filename matches the question's filename, not a document id.\n\nThe lesson evaluates whole FAQ records, while this homework evaluates chunks. Adjust the relevance logic to use the fields available on the chunk results and homework ground truth instead of copying the lesson's record-based comparison unchanged.\n\nAs a reminder, these functions do the following:\n\n- compute_relevance runs search for a question and returns a list of 0s and 1s\n- hit_rate is the fraction of questions where the correct page appears in the results\n- mrr (Mean Reciprocal Rank) also rewards finding the page near the top\n- evaluate runs a search function over the whole ground truth and returns both metrics\n"

In [29]:
chunks[:2]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [30]:
ground_truth[:2]

[{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does this course build the RAG project in plain Python instead of starting with a framework or library?',
  'filename': '01-agentic-rag/lessons/01-intro.md'}]

In [31]:
# RELEVANCE

# generic compute of relevance so we can evaluate vector search, hybrid search, or another retrieval method
# but the relevance logic stays the same 

def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

# for all questions in the ground truth data 

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [32]:
# HIT RATE 

# putting this in a function

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [41]:
# MRR

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [34]:
"""
Q4. Evaluating text search
Evaluate text_search on the ground truth data.

What's the Hit Rate?

0.55
0.66
0.76
0.88
"""


# compute relevance 

relevance_total = compute_relevance_total(ground_truth=ground_truth, search_function=text_search)
relevance_total[:5]

# compute hit rate 

hit_rate(relevance_total)

# 0.7583 -> NEAREST SOLUTION: 0.76



  0%|          | 0/360 [00:00<?, ?it/s]

0.7583333333333333

In [35]:
"""
Q5. Evaluating vector search
Now evaluate vector_search - the part we left for the homework, since the module only evaluated keyword search.

What's the MRR?

0.35
0.45
0.55
0.65
"""

# compute relevance 

relevance_total = compute_relevance_total(ground_truth=ground_truth, search_function=vector_search)
print(relevance_total[:5])







  0%|          | 0/360 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0], [0, 0, 0, 1, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]


In [ ]:
# compute MRR 

mrr = mrr(relevance_total)
print(mrr)

# 0.5486 -> NEAREST SOLUTION: 0.55

0.5486111111111112


In [37]:
"""
Q6. Tuning hybrid search
The k constant in RRF controls how much the top ranks matter. A smaller k sharpens the gap between positions, so being at the top of a list counts for more. The RRF paper uses 60 as a default, but the best value depends on the data

so let's measure it.
Evaluate hybrid_search over the full ground truth dataset for k values 1, 50, 100, and 200. Compare the MRR values for these runs.

Which k gives the best MRR?

1
50
100
200
Several values of k may give the same MRR. If there's a tie, pick the smallest k.


"""

"\nQ6. Tuning hybrid search\nThe k constant in RRF controls how much the top ranks matter. A smaller k sharpens the gap between positions, so being at the top of a list counts for more. The RRF paper uses 60 as a default, but the best value depends on the data\n\nso let's measure it.\nEvaluate hybrid_search over the full ground truth dataset for k values 1, 50, 100, and 200. Compare the MRR values for these runs.\n\nWhich k gives the best MRR?\n\n1\n50\n100\n200\nSeveral values of k may give the same MRR. If there's a tie, pick the smallest k.\n\n\n"

In [44]:
# different k values: 

hybrid_search_mrr_values = {}

for k in [1, 50, 100, 200]:

    # print which k we are at
    print("k: ", k, "\n")

    # perform hybrid search and compute relevance total
    print("Performing compute_relevance_total with hybrid_search on entire ground_truth data ... ")
    relevance_total = compute_relevance_total(ground_truth=ground_truth, search_function=hybrid_search)
    print("Done.")
    print("relevance_total: ", relevance_total)

    # compute MRR 
    print("Compute MRR ... ")
    mrr_value = mrr(relevance_total)
    print("Done.")
    print("MRR: ", mrr_value)

    # append to results
    hybrid_search_mrr_values[k] = mrr_value




k:  1 

Performing compute_relevance_total with hybrid_search on entire ground_truth data ... 


  0%|          | 0/360 [00:00<?, ?it/s]

Done.
relevance_total:  [[1, 0, 0, 0, 0], [1, 0, 0, 0, 1], [1, 0, 0, 1, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 1, 0, 1], [0, 1, 0, 0, 1], [0, 0, 0, 0, 0], [1, 0, 1, 0, 0], [1, 0, 0, 0, 1], [1, 1, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 1, 0, 1], [0, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 1, 0], [0, 1, 0, 0, 0], [1, 1, 1, 0, 0], [0, 1, 1, 0, 0], [1, 1, 0, 1, 0], [0, 0, 0, 0, 1], [0, 0, 0, 1, 0], [1, 1, 1, 1, 1], [1, 1, 1, 0, 0], [1, 0, 1, 0, 0], [1, 0, 1, 0, 0], [0, 0, 0, 0, 0], [1, 1, 0, 0, 0], [1, 1, 0, 1, 0], [0, 0, 1, 1, 0], [1, 1, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 1, 0], [1, 1, 0, 0, 1], [0, 1, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 1, 0], [0, 0, 0, 0, 0], [0, 1, 0, 0, 0], [1, 1, 0, 1, 1], [1, 1, 0, 1, 0], [1, 1, 1, 0, 1], [1, 1, 1, 1, 0], [1, 1, 1, 1, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 1, 1, 0], [1, 0, 1, 0, 0], [1, 0, 1, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0,

  0%|          | 0/360 [00:00<?, ?it/s]

Done.
relevance_total:  [[1, 0, 0, 0, 0], [1, 0, 0, 0, 1], [1, 0, 0, 1, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 1, 0, 1], [0, 1, 0, 0, 1], [0, 0, 0, 0, 0], [1, 0, 1, 0, 0], [1, 0, 0, 0, 1], [1, 1, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 1, 0, 1], [0, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 1, 0], [0, 1, 0, 0, 0], [1, 1, 1, 0, 0], [0, 1, 1, 0, 0], [1, 1, 0, 1, 0], [0, 0, 0, 0, 1], [0, 0, 0, 1, 0], [1, 1, 1, 1, 1], [1, 1, 1, 0, 0], [1, 0, 1, 0, 0], [1, 0, 1, 0, 0], [0, 0, 0, 0, 0], [1, 1, 0, 0, 0], [1, 1, 0, 1, 0], [0, 0, 1, 1, 0], [1, 1, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 1, 0], [1, 1, 0, 0, 1], [0, 1, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 1, 0], [0, 0, 0, 0, 0], [0, 1, 0, 0, 0], [1, 1, 0, 1, 1], [1, 1, 0, 1, 0], [1, 1, 1, 0, 1], [1, 1, 1, 1, 0], [1, 1, 1, 1, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 1, 1, 0], [1, 0, 1, 0, 0], [1, 0, 1, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0,

  0%|          | 0/360 [00:00<?, ?it/s]

Done.
relevance_total:  [[1, 0, 0, 0, 0], [1, 0, 0, 0, 1], [1, 0, 0, 1, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 1, 0, 1], [0, 1, 0, 0, 1], [0, 0, 0, 0, 0], [1, 0, 1, 0, 0], [1, 0, 0, 0, 1], [1, 1, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 1, 0, 1], [0, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 1, 0], [0, 1, 0, 0, 0], [1, 1, 1, 0, 0], [0, 1, 1, 0, 0], [1, 1, 0, 1, 0], [0, 0, 0, 0, 1], [0, 0, 0, 1, 0], [1, 1, 1, 1, 1], [1, 1, 1, 0, 0], [1, 0, 1, 0, 0], [1, 0, 1, 0, 0], [0, 0, 0, 0, 0], [1, 1, 0, 0, 0], [1, 1, 0, 1, 0], [0, 0, 1, 1, 0], [1, 1, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 1, 0], [1, 1, 0, 0, 1], [0, 1, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 1, 0], [0, 0, 0, 0, 0], [0, 1, 0, 0, 0], [1, 1, 0, 1, 1], [1, 1, 0, 1, 0], [1, 1, 1, 0, 1], [1, 1, 1, 1, 0], [1, 1, 1, 1, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 1, 1, 0], [1, 0, 1, 0, 0], [1, 0, 1, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0,

  0%|          | 0/360 [00:00<?, ?it/s]

Done.
relevance_total:  [[1, 0, 0, 0, 0], [1, 0, 0, 0, 1], [1, 0, 0, 1, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 1, 0, 1], [0, 1, 0, 0, 1], [0, 0, 0, 0, 0], [1, 0, 1, 0, 0], [1, 0, 0, 0, 1], [1, 1, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 1, 0, 1], [0, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 0, 1, 0], [0, 1, 0, 0, 0], [1, 1, 1, 0, 0], [0, 1, 1, 0, 0], [1, 1, 0, 1, 0], [0, 0, 0, 0, 1], [0, 0, 0, 1, 0], [1, 1, 1, 1, 1], [1, 1, 1, 0, 0], [1, 0, 1, 0, 0], [1, 0, 1, 0, 0], [0, 0, 0, 0, 0], [1, 1, 0, 0, 0], [1, 1, 0, 1, 0], [0, 0, 1, 1, 0], [1, 1, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 1, 0], [1, 1, 0, 0, 1], [0, 1, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 1, 0], [0, 0, 0, 0, 0], [0, 1, 0, 0, 0], [1, 1, 0, 1, 1], [1, 1, 0, 1, 0], [1, 1, 1, 0, 1], [1, 1, 1, 1, 0], [1, 1, 1, 1, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 1, 1, 0], [1, 0, 1, 0, 0], [1, 0, 1, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0,

In [45]:
hybrid_search_mrr_values

{1: 0.637916666666667,
 50: 0.637916666666667,
 100: 0.637916666666667,
 200: 0.637916666666667}

In [ ]:
# sort dictionary: Which k gives the best MRR?

sorted_k = sorted(
    hybrid_search_mrr_values,
    key=hybrid_search_mrr_values.get,
    reverse=True    
)

sorted_k

# -> SOLUTION with: Several values of k may give the same MRR. If there's a tie, pick the smallest k. 
# -> 1

[1, 50, 100, 200]